# Notebook 4 — HMM K=4 + Viterbi

**Inputs:** `outputs/trajectories_full.csv`, `outputs/clusters_k4.csv`, `data/grades_export_anon.csv`  
**Outputs:** `outputs/hmm_k4_hidden_states_long.csv`, `outputs/hmm_k4_hidden_states_summary.csv`

Requiere: `pip install hmmlearn`

In [ ]:
import os, numpy as np, random, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from hmmlearn import hmm
from scipy.stats import spearmanr
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
BASE_PATH = '.'
INPUTS  = os.path.join(BASE_PATH, 'data')
OUTPUTS = os.path.join(BASE_PATH, 'outputs')
ID_COL = 'email'; STATE_COL = 'state'; TIME_COL = 'timestamp'
print(f'INPUTS: {INPUTS} | OUTPUTS: {OUTPUTS} | SEED: {SEED}')


In [ ]:
traj_full = pd.read_csv(f"{OUTPUTS}/trajectories_full.csv")
clusters  = pd.read_csv(f"{OUTPUTS}/clusters_k4.csv")
grades    = pd.read_csv(f"{INPUTS}/grades_export_anon.csv")
grades.columns = [c.strip().lower() for c in grades.columns]
traj_full.head()


In [ ]:
# Codificación de estados observables
state_map = {s:i for i,s in enumerate(traj_full[STATE_COL].unique())}
inv_state_map = {v:k for k,v in state_map.items()}
traj_full['obs'] = traj_full[STATE_COL].map(state_map)
print('Estados observables:', state_map)


In [ ]:
# Construcción de secuencias por estudiante
traj_full = traj_full.sort_values([ID_COL, TIME_COL])
sequences, lengths, emails_order = [], [], []
for email, g in traj_full.groupby(ID_COL):
    sequences.append(g['obs'].values)
    lengths.append(len(g))
    emails_order.append(email)
X = np.concatenate(sequences).reshape(-1, 1)
print(f'X shape: {X.shape}, alumnos: {len(lengths)}')


In [ ]:
# Entrenamiento HMM K=4
model = hmm.CategoricalHMM(n_components=4, n_iter=1000, random_state=SEED)
model.fit(X, lengths)
ll = model.score(X, lengths)
print(f'Log-likelihood: {ll:.4f}')


In [ ]:
# Matriz de emisión B
emission_df = pd.DataFrame(
    model.emissionprob_,
    columns=[inv_state_map[i] for i in range(len(state_map))]
)
print('Matriz de emisión B:')
print(emission_df.round(3))


In [ ]:
# Decodificación Viterbi
records = []
for email, seq in zip(emails_order, sequences):
    z = model.predict(seq.reshape(-1, 1))
    for t, zi in enumerate(z):
        records.append({ID_COL: email, 't': t, 'Z': zi})
z_long = pd.DataFrame(records)
z_long.to_csv(f"{OUTPUTS}/hmm_k4_hidden_states_long.csv", index=False)

# Proporciones Z por alumno
z_summary = z_long.groupby([ID_COL,'Z']).size().unstack(fill_value=0)
z_prop = z_summary.div(z_summary.sum(axis=1), axis=0)
z_prop.columns = [f'Z{c}' for c in z_prop.columns]
z_prop = z_prop.reset_index()
z_prop.to_csv(f"{OUTPUTS}/hmm_k4_hidden_states_summary.csv", index=False)
print(f'Viterbi OK — z_long: {len(z_long)} filas')
z_prop.round(3)


In [ ]:
# Correlaciones Z vs nota_final
GRADE_COL = next((c for c in ['nota_final','nota','final_grade'] if c in grades.columns), None)
Z_COLS = [c for c in z_prop.columns if c.startswith('Z')]
df_z = z_prop.merge(grades[['email', GRADE_COL]], on='email')
print(f'Correlaciones Spearman Z vs {GRADE_COL} (n={len(df_z)}):')
for z in Z_COLS:
    rho, p = spearmanr(df_z[z], df_z[GRADE_COL])
    sig = '**' if p<0.05 else ('*' if p<0.10 else 'ns')
    print(f'  {z}: rho={rho:+.3f}, p={p:.4f}  {sig}')
